<a href="https://colab.research.google.com/github/enyachang1119-debug/DS2002/blob/main/2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [7]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [8]:
# TODO
removed = df.duplicated().sum() # Count the number of exact duplicate rows
clean = df.drop_duplicates().copy() # Remove exact duplicates and create a separate copy of the cleaned dataframe
log('duplicates', 'dropped exact duplicate rows', removed) # Log the cleaning decision and number of rows removed

[duplicates] dropped exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [9]:
# TODO
clean['price'] = clean['price'].str.replace('$', '', regex=False).str.strip().astype(float) # Remove dollar signs and whitespace, then convert price to float
assert clean['price'].dtype == float # Check that price was successfully converted to float
log('price', 'removed dollar signs and whitespace, then converted price to float', len(clean)) # Log the cleaning decision and number of rows affected


[price] removed dollar signs and whitespace, then converted price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [10]:
# TODO
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce') # Convert quantity to numeric

# Count rows with missing and negative quantities
missing_qty = clean['qty'].isna().sum()
negative_qty = (clean['qty'] < 0).sum()

clean = clean[clean['qty'].notna() & (clean['qty'] >= 0)].copy() # Drop rows with missing or negative quantities

# Log the number of rows removed for each reason
log('quantity_missing', 'dropped rows with missing quantity', missing_qty)
log('quantity_negative', 'dropped rows with negative quantity', negative_qty)

[quantity_missing] dropped rows with missing quantity (12 row(s))
[quantity_negative] dropped rows with negative quantity (13 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [12]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(clean['item'].value_counts()) # Inspect the different item name variants

before = clean['item'].nunique() # Count the number of item variants before mapping
ITEM_MAP = {
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho',
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger'
} # Map the six spellings into three consistent product names

clean['item'] = clean['item'].replace(ITEM_MAP) # Apply the mapping to the item column
after = clean['item'].nunique() # Count the number of unique items after mapping
log('item', f'canonicalized item names from {before} variants to {after} products', before - after) # Log how many item variants were collapsed

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] canonicalized item names from 6 variants to 3 products (3 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [14]:
# TODO
print(clean['category'].value_counts())
before = clean['category'].nunique() # Count the number of category variants before mapping

CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear',
    'Merch': 'Merch',
    'Apparel': 'Merch'
} # Map category variants into consistent category names

clean['category'] = clean['category'].replace(CATEGORY_MAP) # Apply the mapping to the category column
after = clean['category'].nunique() # Count the number of categories after mapping
log('category', f'normalized categories from {before} variants to {after} categories; combined Apparel with Merch', before - after) # Log the business decision and category collapse

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] normalized categories from 6 variants to 3 categories; combined Apparel with Merch (3 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [21]:
assert clean.duplicated().sum() == 0
assert clean['qty'].min() >= 1
assert clean['price'].dtype == float
# Check that there are three unique item names
assert clean['item'].nunique() == 3
# Check that there are three unique categories
assert clean['category'].nunique() == 3
print('clean:', clean.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [22]:
# TODO
clean['revenue'] = clean['qty'] * clean['price'] # Calculate revenue for each row
revenue_by_category = clean.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2) # Calculate revenue by category and sort from highest to lowest

# Calculate the overall revenue
total_revenue = clean['revenue'].sum().round(2) # Calculate the overall revenue

# Print revenue by category and the overall total
print('Revenue by category:')
print(revenue_by_category)
print('Overall revenue:', total_revenue)

Revenue by category:
category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Overall revenue: 4740.0


**What I would tell the vendor:** _I would tell the vendor to stock more Food because it has the highest revenue, followed by Merch and RainGear_

### TODO 8 — read back your log

In [23]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped exact duplicate rows,15
1,duplicates,dropped exact duplicate rows,15
2,duplicates,dropped exact duplicate rows,15
3,price,"removed dollar signs and whitespace, then conv...",300
4,quantity_missing,dropped rows with missing quantity,12
5,quantity_negative,dropped rows with negative quantity,13
6,item,canonicalized item names from 6 variants to 3 ...,3
7,category,normalized categories from 6 variants to 3 cat...,3


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.


_a: Dropping rows with negative quantities changed the revenue total the most, from $4,594.50 to $4,740.00._

_b: I chose to drop rows with negative quantities. A reasonable alternative would be to keep them and treat them as refunds, which would reduce the reported revenue from $4,740.00 to $4,594.50. I dropped them because this analysis treats negative quantities as invalid sales records and focuses on revenue from completed sales._
